### Reguläre Sprachen

[Video1](https://youtu.be/weBEcX8j-nA) - [Video2](https://youtu.be/3Y5EijPeFaY) - [Aufgabenvideos](https://www.youtube.com/playlist?list=PLWeMgMhRDsIGeEHYyziNcWWaJHQzQKQNS)

Dieses Notebook setzt das Notebook **Endliche Automaten** voraus: deterministische endliche Automaten (DEA) mit
Zustandsdiagramm und Zustandsübergangstabelle, ihre Simulation in Python und Mealy-Automaten. Hier geht es um die
Frage, welche **Sprachen** endliche Automaten erkennen können:

- formale Sprachen,
- nichtdeterministische endliche Automaten (NEA) und ihre Umwandlung in DEAs,
- reguläre Ausdrücke,
- die Grenzen endlicher Automaten (Pumping-Lemma).

### Alphabet, Wort, Sprache

- Ein **Alphabet** $\Sigma$ ist eine endliche Menge von Zeichen, z.B. $\Sigma = \{0, 1\}$.
- Ein **Wort** über $\Sigma$ ist eine endliche Folge von Zeichen aus $\Sigma$, z.B. $w = 010010$. Die Anzahl der
  Zeichen ist die **Länge** $|w|$, hier $|w| = 6$.
- Das **leere Wort** $\varepsilon$ hat die Länge 0.
- $\Sigma^*$ ist die Menge **aller** Wörter über $\Sigma$, einschließlich $\varepsilon$. Sie ist unendlich.
- Eine (formale) **Sprache** $L$ über $\Sigma$ ist eine Teilmenge von $\Sigma^*$.

Beispiel: die Sprache der Wörter über $\{0, 1\}$, in denen keine zwei Einsen aufeinanderfolgen,
$L = \{\varepsilon, 0, 1, 00, 01, 10, 000, 001, 010, 100, 101, \dots\}$. Mit Python erzeugen wir alle Wörter bis zur
Länge 6 und zählen, wie viele davon in $L$ liegen:

In [1]:
import itertools

def woerter(alphabet, laenge):
    return [''.join(t) for t in itertools.product(alphabet, repeat=laenge)]

print(woerter('01', 2))
for n in range(7):
    alle = woerter('01', n)
    in_L = [w for w in alle if '11' not in w]
    print(n, len(alle), len(in_L))

['00', '01', '10', '11']
0 1 1
1 2 2
2 4 3
3 8 5
4 16 8
5 32 13
6 64 21


Über einem Alphabet mit 2 Zeichen gibt es $2^n$ Wörter der Länge $n$. Davon liegen 1, 2, 3, 5, 8, 13, 21 in $L$,
jede Zahl ist die Summe der beiden vorherigen (Fibonacci-Zahlen). Eine Sprache kann also unendlich viele Wörter
enthalten und trotzdem durch eine einfache Regel beschrieben werden.

Die Unterscheidung von **Syntax** und **Semantik** gilt auch hier: Eine formale Sprache legt nur fest, welche
Zeichenfolgen dazugehören (Syntax), nicht, was sie bedeuten (Semantik). Wie man Sprachen mit Grammatiken beschreibt,
steht im Notebook **Grammatiken**.

### Die Sprache eines DEA

Ein DEA **akzeptiert** ein Wort, wenn er nach dem Lesen aller Zeichen in einem Endzustand ist. Die Menge aller
Wörter, die ein DEA $A$ akzeptiert, ist die **Sprache** $L(A)$ des Automaten. Der folgende DEA akzeptiert genau die
Wörter ohne zwei aufeinanderfolgende Einsen. $q_2$ ist ein Fehlerzustand: Wer dort ankommt, bleibt dort.

<img src="dea_keine11.png" width="380">

Wie im Notebook Endliche Automaten speichern wir die Übergänge in einem Dictionary. Damit wir später auch
nichtdeterministische Automaten simulieren können, ist jeder Wert gleich eine **Menge** von Folgezuständen, bei einem
DEA mit genau einem Element.

In [2]:
keine11 = {
    'start': 'q0',
    'ende': {'q0', 'q1'},
    'delta': {('q0', '0'): {'q0'}, ('q0', '1'): {'q1'},
              ('q1', '0'): {'q0'}, ('q1', '1'): {'q2'},
              ('q2', '0'): {'q2'}, ('q2', '1'): {'q2'}},
}

def dea_akzeptiert(aut, wort):
    zustand = aut['start']
    for zeichen in wort:
        (zustand,) = aut['delta'][(zustand, zeichen)]
    return zustand in aut['ende']

print([w for n in range(5) for w in woerter('01', n) if dea_akzeptiert(keine11, w)])

['', '0', '1', '00', '01', '10', '000', '001', '010', '100', '101', '0000', '0001', '0010', '0100', '0101', '1000', '1001', '1010']


`(zustand,) = ...` packt die Menge mit einem Element aus. Die akzeptierten Wörter bis zur Länge 4 sind genau die
ohne `11`.

**Das Komplement:** Vertauscht man bei einem DEA Endzustände und Nicht-Endzustände, akzeptiert er genau die Wörter,
die er vorher abgelehnt hat. Dafür muss es für jeden Zustand und jedes Zeichen einen Übergang geben, notfalls in
einen Fehlerzustand. Sonst würden Wörter, bei denen der DEA hängen bleibt, weder vorher noch nachher akzeptiert.

In [3]:
mit11 = dict(keine11, ende={'q2'})
print([w for n in range(5) for w in woerter('01', n) if dea_akzeptiert(mit11, w)])

['11', '011', '110', '111', '0011', '0110', '0111', '1011', '1100', '1101', '1110', '1111']


### Nichtdeterministische endliche Automaten (NEA)

Bei einem **nichtdeterministischen endlichen Automaten** (NEA) darf es zu einem Zustand und einem Zeichen mehrere
Folgezustände geben, oder auch keinen. Ein NEA **akzeptiert** ein Wort, wenn es **mindestens einen** Weg durch den
Automaten gibt, der nach dem Lesen aller Zeichen in einem Endzustand endet.

Beispiel: alle Wörter über $\{0, 1\}$, die $010$ enthalten.

<img src="nea_010.png" width="420">

Im Zustand A kann der NEA nach einer 0 in A bleiben oder nach B gehen. Man kann sich vorstellen, dass er **rät**, wo
die Folge $010$ beginnt. Rät er richtig, kommt er nach D. Ein NEA ist oft viel leichter zu entwerfen als ein DEA:
Hier schreibt man das gesuchte Muster einfach hin.

Simulieren kann man einen NEA, indem man sich die **Menge** aller Zustände merkt, in denen er gerade sein könnte:

In [4]:
nea010 = {
    'start': 'A',
    'ende': {'D'},
    'delta': {('A', '0'): {'A', 'B'}, ('A', '1'): {'A'},
              ('B', '1'): {'C'},
              ('C', '0'): {'D'},
              ('D', '0'): {'D'}, ('D', '1'): {'D'}},
}

def nea_akzeptiert(aut, wort, ausgabe=False):
    aktuell = {aut['start']}
    for zeichen in wort:
        neu = set()
        for q in aktuell:
            neu |= aut['delta'].get((q, zeichen), set())
        if ausgabe:
            print(sorted(aktuell), zeichen, '->', sorted(neu))
        aktuell = neu
    return len(aktuell & aut['ende']) > 0

print(nea_akzeptiert(nea010, '1001010', ausgabe=True))
print(nea_akzeptiert(nea010, '0110'))

['A'] 1 -> ['A']
['A'] 0 -> ['A', 'B']
['A', 'B'] 0 -> ['A', 'B']
['A', 'B'] 1 -> ['A', 'C']
['A', 'C'] 0 -> ['A', 'B', 'D']
['A', 'B', 'D'] 1 -> ['A', 'C', 'D']
['A', 'C', 'D'] 0 -> ['A', 'B', 'D']
True
False


`aut['delta'].get((q, zeichen), set())` liefert die leere Menge, wenn es keinen Übergang gibt. Beim Wort `1001010`
kommt die Menge nach dem fünften Zeichen zum ersten Mal an D an, und D bleibt danach immer dabei.

#### $\varepsilon$-Übergänge

Ein NEA darf außerdem **$\varepsilon$-Übergänge** haben: Er wechselt den Zustand, ohne ein Zeichen zu lesen. Damit
kann man Automaten leicht zusammensetzen. Der folgende NEA akzeptiert alle Wörter, die nur aus a bestehen, oder aus
beliebig vielen ab hintereinander:

<img src="nea_eps.png" width="330">

Bei der Simulation muss man nach jedem Schritt (und am Anfang) alle Zustände hinzunehmen, die über
$\varepsilon$-Übergänge erreichbar sind, die **$\varepsilon$-Hülle**. Das $\varepsilon$ schreiben wir im Code als
leeren String.

In [5]:
nea_eps = {
    'start': 's',
    'ende': {'p', 'r'},
    'delta': {('s', ''): {'p', 'r'}, ('p', 'a'): {'p'}, ('r', 'a'): {'t'}, ('t', 'b'): {'r'}},
}

def huelle(aut, menge):
    stapel = list(menge)
    menge = set(menge)
    while stapel:
        q = stapel.pop()
        for r in aut['delta'].get((q, ''), set()):
            if r not in menge:
                menge.add(r)
                stapel.append(r)
    return menge

def akzeptiert(aut, wort):
    aktuell = huelle(aut, {aut['start']})
    for zeichen in wort:
        neu = set()
        for q in aktuell:
            neu |= aut['delta'].get((q, zeichen), set())
        aktuell = huelle(aut, neu)
    return len(aktuell & aut['ende']) > 0

for w in ['', 'aaa', 'abab', 'aab', 'aba']:
    print(repr(w), akzeptiert(nea_eps, w))

'' True
'aaa' True
'abab' True
'aab' False
'aba' False


### Vom NEA zum DEA: die Potenzmengenkonstruktion

Die Simulation zeigt schon den Trick: Wenn man die **Menge** der möglichen Zustände als einen einzigen Zustand
auffasst, wird aus dem NEA ein DEA. Jeder NEA lässt sich so in einen DEA umwandeln, der dieselbe Sprache akzeptiert.
Das Verfahren heißt **Potenzmengenkonstruktion** (die Potenzmenge ist die Menge aller Teilmengen).

1. Der Startzustand des DEA ist die Menge $\{A\}$ (mit $\varepsilon$-Hülle).
2. Für jede neue Menge und jedes Zeichen bestimmt man die Menge der Folgezustände.
3. Das wiederholt man, bis keine neuen Mengen mehr entstehen.
4. Endzustände des DEA sind alle Mengen, die einen Endzustand des NEA enthalten.

Für den NEA mit $010$:

| | 0 | 1 |
|---|---|---|
| $\rightarrow \{A\}$ | $\{A, B\}$ | $\{A\}$ |
| $\{A, B\}$ | $\{A, B\}$ | $\{A, C\}$ |
| $\{A, C\}$ | $\{A, B, D\}$ | $\{A\}$ |
| $* \{A, B, D\}$ | $\{A, B, D\}$ | $\{A, C, D\}$ |
| $* \{A, C, D\}$ | $\{A, B, D\}$ | $\{A, D\}$ |
| $* \{A, D\}$ | $\{A, B, D\}$ | $\{A, D\}$ |

<img src="dea_010.png" width="700">

In [6]:
def potenzmengen(aut, alphabet):
    start = frozenset(huelle(aut, {aut['start']}))
    delta = {}
    offen = [start]
    bekannt = [start]
    while offen:
        menge = offen.pop(0)
        for zeichen in alphabet:
            neu = set()
            for q in menge:
                neu |= aut['delta'].get((q, zeichen), set())
            neu = frozenset(huelle(aut, neu))
            delta[(menge, zeichen)] = {neu}
            if neu not in bekannt:
                bekannt.append(neu)
                offen.append(neu)
    ende = {m for m in bekannt if m & aut['ende']}
    return {'start': start, 'ende': ende, 'delta': delta}, bekannt

dea010, mengen = potenzmengen(nea010, '01')
for m in mengen:
    print(sorted(m), [sorted(next(iter(dea010['delta'][(m, z)]))) for z in '01'], '*' if m in dea010['ende'] else '')

['A'] [['A', 'B'], ['A']] 
['A', 'B'] [['A', 'B'], ['A', 'C']] 
['A', 'C'] [['A', 'B', 'D'], ['A']] 
['A', 'B', 'D'] [['A', 'B', 'D'], ['A', 'C', 'D']] *
['A', 'C', 'D'] [['A', 'B', 'D'], ['A', 'D']] *
['A', 'D'] [['A', 'B', 'D'], ['A', 'D']] *


Zur Kontrolle vergleichen wir NEA und DEA für alle Wörter bis zur Länge 12:

In [7]:
alle = [w for n in range(13) for w in woerter('01', n)]
print(len(alle), all(akzeptiert(nea010, w) == dea_akzeptiert(dea010, w) for w in alle))

8191 True


Das Ergebnis muss nicht der kleinste DEA sein. Hier sind die drei Endzustände gleichwertig: Sobald $010$ gelesen
wurde, wird jedes Wort akzeptiert. Fasst man sie zu einem Zustand D zusammen, bleiben vier Zustände:

<img src="dea_010_min.png" width="430">

Die Potenzmengenkonstruktion kann im schlimmsten Fall sehr viele Zustände erzeugen: Ein NEA mit $n$ Zuständen hat
$2^n$ Teilmengen. Ein Beispiel ist die Sprache „das $k$-te Zeichen von hinten ist eine 1“. Der NEA rät, wann er an
dieser Stelle ist, und braucht dafür $k + 1$ Zustände:

In [8]:
def k_von_hinten(k):
    delta = {(0, '0'): {0}, (0, '1'): {0, 1}}
    for i in range(1, k):
        delta[(i, '0')] = {i + 1}
        delta[(i, '1')] = {i + 1}
    return {'start': 0, 'ende': {k}, 'delta': delta}

for k in range(1, 9):
    dea, mengen = potenzmengen(k_von_hinten(k), '01')
    print(k, 'NEA:', k + 1, 'Zustände   DEA:', len(mengen), 'Zustände')

1 NEA: 2 Zustände   DEA: 2 Zustände
2 NEA: 3 Zustände   DEA: 4 Zustände
3 NEA: 4 Zustände   DEA: 8 Zustände
4 NEA: 5 Zustände   DEA: 16 Zustände
5 NEA: 6 Zustände   DEA: 32 Zustände
6 NEA: 7 Zustände   DEA: 64 Zustände
7 NEA: 8 Zustände   DEA: 128 Zustände
8 NEA: 9 Zustände   DEA: 256 Zustände


Der DEA muss sich die letzten $k$ Zeichen merken und braucht dafür $2^k$ Zustände. Weniger geht in diesem Fall auch
nicht. NEA und DEA erkennen also dieselben Sprachen, der NEA ist aber oft viel kleiner.

### Reguläre Sprachen und reguläre Ausdrücke

Eine Sprache heißt **regulär**, wenn es einen DEA gibt, der genau diese Sprache akzeptiert.

Reguläre Sprachen lassen sich auch ohne Automaten beschreiben, mit **regulären Ausdrücken**. Sie werden aus den
Zeichen des Alphabets und $\varepsilon$ gebildet mit

- **Vereinigung** $r \cup s$: ein Wort aus $r$ oder aus $s$,
- **Konkatenation** $rs$: ein Wort aus $r$, dahinter ein Wort aus $s$,
- **Kleene-Stern** $r^*$: beliebig viele (auch 0) Wörter aus $r$ hintereinander,
- Klammern.

Ohne Klammern bindet $^*$ am stärksten, dann die Konkatenation, dann $\cup$.

| Ausdruck über $\{a, b\}$ | Sprache |
|---|---|
| $ab \cup aab$ | $\{ab, aab\}$ |
| $a(b \cup a)aab$ | $\{abaab, aaaab\}$ |
| $(ab)^*$ | $\varepsilon$, $ab$, $abab$, ... |
| $b^*b$ | alle Wörter aus mindestens einem b |
| $(a \cup b)^*aa(a \cup b)^*$ | alle Wörter, die $aa$ enthalten |
| $(a \cup ba)^*(\varepsilon \cup b)$ | alle Wörter, in denen keine zwei b aufeinanderfolgen |

#### Reguläre Ausdrücke in Python

In der Praxis benutzt man reguläre Ausdrücke zum Suchen und Prüfen von Texten. Python hat dafür das Modul `re`. Die
Schreibweise ist etwas anders:

| Theorie | Python | Bedeutung |
|---|---|---|
| $a \cup b$ | `a\|b` | a oder b |
| $(ab)^*$ | `(ab)*` | beliebig oft, auch 0-mal |
| $aa^*$ | `a+` | mindestens einmal |
| $\varepsilon \cup a$ | `a?` | höchstens einmal |
| $a \cup b \cup c$ | `[abc]` | eines der Zeichen |

`re.fullmatch(muster, wort)` prüft, ob das **ganze** Wort passt. Damit überprüfen wir die Tabelle oben für alle
Wörter bis zur Länge 8:

In [9]:
import re

beispiele = [
    ('ab|aab', lambda w: w in ('ab', 'aab')),
    ('a(b|a)aab', lambda w: w in ('abaab', 'aaaab')),
    ('(ab)*', lambda w: w == 'ab' * (len(w) // 2)),
    ('b*b', lambda w: len(w) > 0 and set(w) == {'b'}),
    ('(a|b)*aa(a|b)*', lambda w: 'aa' in w),
    ('(a|ba)*b?', lambda w: 'bb' not in w),
]
alle = [w for n in range(9) for w in woerter('ab', n)]
for muster, eigenschaft in beispiele:
    passt = all((re.fullmatch(muster, w) is not None) == eigenschaft(w) for w in alle)
    print(f'{muster:16} {passt}')

ab|aab           True
a(b|a)aab        True
(ab)*            True
b*b              True
(a|b)*aa(a|b)*   True
(a|ba)*b?        True


Mit `re.search` und `re.findall` sucht man ein Muster irgendwo in einem Text, z.B. alle Postleitzahlen. `\d` steht
für eine Ziffer, `{5}` für genau fünf Wiederholungen und `\b` für eine Wortgrenze:

In [10]:
text = 'Hauptstraße 5, 79098 Freiburg; Postfach 1234, 10115 Berlin'
print(re.findall(r'\b\d{5}\b', text))

['79098', '10115']


### Drei Beschreibungen derselben Sprachen

Der **Satz von Kleene** sagt: DEAs, NEAs (auch mit $\varepsilon$-Übergängen) und reguläre Ausdrücke beschreiben
genau dieselben Sprachen, die regulären Sprachen. Dazu kommen die **regulären Grammatiken** (siehe Notebook
Grammatiken).

Die Umwandlungen kann man angeben:

- NEA $\to$ DEA: Potenzmengenkonstruktion.
- Regulärer Ausdruck $\to$ NEA: Man baut den NEA aus Bausteinen zusammen. Für $r \cup s$ verzweigt man vom Start mit
  zwei $\varepsilon$-Übergängen in die Automaten für $r$ und $s$, für $rs$ verbindet man die Endzustände von $r$ mit
  $\varepsilon$ mit dem Start von $s$, für $r^*$ führt man mit $\varepsilon$ zurück zum Start.
- DEA $\to$ regulärer Ausdruck: Man liest die Wege durch den Automaten ab, z.B. beim DEA ohne $11$:
  $(0 \cup 10)^*(\varepsilon \cup 1)$.

In [11]:
print(all((re.fullmatch(r'(0|10)*(|1)', w) is not None) == ('11' not in w)
          for n in range(13) for w in woerter('01', n)))

True


### Abschlusseigenschaften

Sind $L_1$ und $L_2$ regulär, dann sind auch regulär:

- das **Komplement** $\overline{L_1}$ (Endzustände tauschen),
- die **Vereinigung** $L_1 \cup L_2$ (regulärer Ausdruck $r_1 \cup r_2$),
- die **Konkatenation** $L_1 L_2$ und der **Stern** $L_1^*$,
- der **Durchschnitt** $L_1 \cap L_2$.

Für den Durchschnitt baut man einen **Produktautomaten**: Er simuliert beide DEAs gleichzeitig, seine Zustände sind
Paare aus je einem Zustand der beiden DEAs. Beispiel: $L_1$ = gerade Anzahl a, $L_2$ = endet auf b. Der erste DEA hat
die Zustände g (gerade) und u (ungerade), der zweite 0 (endet nicht auf b) und 1 (endet auf b). Endzustand ist das
Paar, bei dem beide in einem Endzustand sind, also g1.

<img src="produkt.png" width="430">

In [12]:
def produkt(a1, a2, alphabet):
    start = (a1['start'], a2['start'])
    delta, offen, bekannt = {}, [start], {start}
    while offen:
        p, q = offen.pop()
        for z in alphabet:
            (p2,) = a1['delta'][(p, z)]
            (q2,) = a2['delta'][(q, z)]
            delta[((p, q), z)] = {(p2, q2)}
            if (p2, q2) not in bekannt:
                bekannt.add((p2, q2))
                offen.append((p2, q2))
    ende = {(p, q) for p, q in bekannt if p in a1['ende'] and q in a2['ende']}
    return {'start': start, 'ende': ende, 'delta': delta}

gerade_a = {'start': 'g', 'ende': {'g'},
            'delta': {('g', 'a'): {'u'}, ('g', 'b'): {'g'}, ('u', 'a'): {'g'}, ('u', 'b'): {'u'}}}
endet_b = {'start': '0', 'ende': {'1'},
           'delta': {('0', 'a'): {'0'}, ('0', 'b'): {'1'}, ('1', 'a'): {'0'}, ('1', 'b'): {'1'}}}
beide = produkt(gerade_a, endet_b, 'ab')
alle = [w for n in range(11) for w in woerter('ab', n)]
print(beide['ende'], all(dea_akzeptiert(beide, w) == (w.count('a') % 2 == 0 and w.endswith('b')) for w in alle))

{('g', '1')} True


### Grenzen: das Pumping-Lemma

Ein DEA hat nur endlich viele Zustände. Er kann sich deshalb nur endlich viel merken. Kann ein DEA die Sprache
$\{0^n 1^n \mid n \ge 0\}$ erkennen, also $n$ Nullen gefolgt von gleich vielen Einsen? Er müsste sich beim Lesen der
Nullen ihre Anzahl merken, und die kann beliebig groß werden.

Genauer: Hat ein DEA $p$ Zustände und liest er ein Wort mit mindestens $p$ Zeichen, dann durchläuft er $p + 1$
Zustände (den Startzustand mitgezählt). Mindestens ein Zustand kommt also doppelt vor (Schubfachprinzip). Zwischen den
beiden Besuchen liest er ein Teilwort $y$ und kehrt in denselben Zustand zurück. Dieses Teilwort kann man weglassen
oder beliebig oft wiederholen, der DEA landet am Ende im selben Zustand wie vorher.

Wir zeigen das am DEA für $010$ mit 4 Zuständen und dem Wort $1010$:

In [13]:
def zerlege(aut, wort):
    zustand = aut['start']
    besucht = {zustand: 0}
    for i, zeichen in enumerate(wort):
        (zustand,) = aut['delta'][(zustand, zeichen)]
        if zustand in besucht:
            j = besucht[zustand]
            return wort[:j], wort[j:i + 1], wort[i + 1:]
        besucht[zustand] = i + 1

min010 = {'start': 'A', 'ende': {'D'},
          'delta': {('A', '0'): {'AB'}, ('A', '1'): {'A'}, ('AB', '0'): {'AB'}, ('AB', '1'): {'AC'},
                    ('AC', '0'): {'D'}, ('AC', '1'): {'A'}, ('D', '0'): {'D'}, ('D', '1'): {'D'}}}
x, y, z = zerlege(min010, '1010')
print('x =', repr(x), ' y =', repr(y), ' z =', repr(z))
for i in range(4):
    print(i, x + y * i + z, dea_akzeptiert(min010, x + y * i + z))

x = ''  y = '1'  z = '010'
0 010 True
1 1010 True
2 11010 True
3 111010 True


Schon nach dem ersten Zeichen ist der DEA wieder in A. Also ist $x = \varepsilon$, $y = 1$, $z = 010$, und alle
Wörter $1^i 010$ werden akzeptiert. Daraus folgt das **Pumping-Lemma**:

> Ist $L$ regulär, dann gibt es eine Zahl $p$ (z.B. die Zahl der Zustände eines DEA für $L$), so dass sich jedes Wort
> $w \in L$ mit $|w| \ge p$ zerlegen lässt in $w = xyz$ mit
> 1. $|xy| \le p$,
> 2. $|y| \ge 1$,
> 3. $xy^iz \in L$ für alle $i \ge 0$.

Die Bedingung $|xy| \le p$ gilt, weil die Wiederholung spätestens nach $p$ Zeichen auftritt.

#### Anwendung: Eine Sprache ist nicht regulär

**Behauptung:** $L = \{0^n 1^n \mid n \ge 0\}$ ist nicht regulär.

**Beweis:** Angenommen, $L$ ist regulär. Dann gibt es eine Zahl $p$ wie im Pumping-Lemma. Wir wählen
$w = 0^p 1^p \in L$, es gilt $|w| = 2p \ge p$. Sei $w = xyz$ irgendeine Zerlegung mit $|xy| \le p$ und $|y| \ge 1$.
Weil die ersten $p$ Zeichen von $w$ Nullen sind, besteht $y$ nur aus Nullen, etwa $y = 0^k$ mit $k \ge 1$. Dann ist
$xy^2z = 0^{p+k}1^p$. Dieses Wort hat mehr Nullen als Einsen und liegt nicht in $L$. Das widerspricht dem
Pumping-Lemma. Also ist $L$ nicht regulär.

Wichtig: Man darf $w$ wählen (geschickt, abhängig von $p$), aber nicht die Zerlegung. Der Widerspruch muss für
**jede** erlaubte Zerlegung entstehen. Und: Das Pumping-Lemma kann nur zeigen, dass eine Sprache **nicht** regulär ist.
Erfüllt eine Sprache die Bedingungen, ist sie deshalb noch nicht regulär.

#### Was endliche Automaten nicht können

Nicht regulär sind z.B.

- $\{0^n 1^n\}$ und korrekt geklammerte Ausdrücke wie `(()(()))`: Man müsste beliebig weit zählen.
- Palindrome und Wörter der Form $ww$: Man müsste sich die erste Hälfte merken.

Mit einem zusätzlichen Speicher kommt man weiter:

| Modell | Speicher | erkennt z.B. |
|---|---|---|
| DEA, NEA | nur der aktuelle Zustand | gerade Anzahl a, Muster wie $010$ |
| Kellerautomat | Zustand und Stapel | $\{0^n 1^n\}$, korrekte Klammerung, Palindrome |
| Turingmaschine | Zustand und unbegrenztes Band | alles, was ein Computer berechnen kann |

Mehr dazu in den Notebooks **Kellerautomaten** und **Turingmaschinen**.

### Zusammenfassung

- Eine **Sprache** ist eine Menge von Wörtern über einem Alphabet. Die Sprache $L(A)$ eines Automaten ist die Menge der
  Wörter, die er akzeptiert.
- Ein **NEA** darf mehrere oder keine Folgezustände haben und $\varepsilon$-Übergänge. Er akzeptiert, wenn es
  mindestens einen Weg zu einem Endzustand gibt. Simulieren kann man ihn mit Mengen von Zuständen.
- Die **Potenzmengenkonstruktion** macht aus jedem NEA einen DEA, dessen Zustände Mengen von NEA-Zuständen sind. Das
  kann bis zu $2^n$ Zustände ergeben.
- **Reguläre Sprachen** werden von DEAs, NEAs, regulären Ausdrücken und regulären Grammatiken beschrieben (Satz von
  Kleene). Sie sind abgeschlossen unter Komplement, Vereinigung, Durchschnitt, Konkatenation und Stern.
- Das **Pumping-Lemma** zeigt, dass manche Sprachen nicht regulär sind, z.B. $\{0^n 1^n\}$: Ein endlicher Automat kann
  nicht beliebig weit zählen.